In [ ]:
import re
from collections import Counter, defaultdict

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
model_name = "bert-base-cased-finetuned-mrpc"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()
print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")


In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples before filtering: {len(dataset)}")
print("Example row:")
print(dataset[0])


In [ ]:
negation_pattern = re.compile(r"\b(?:not|never|no|without)\b|n['’]t\b", flags=re.IGNORECASE)

def extract_negation_cues(text):
    return [m.group(0).lower() for m in negation_pattern.finditer(text)]

def sentence_negation_pattern(cues1, cues2):
    has1 = len(cues1) > 0
    has2 = len(cues2) > 0
    if has1 and has2:
        return "both_sentences_negation"
    if has1 and not has2:
        return "sentence1_only_negation"
    if has2 and not has1:
        return "sentence2_only_negation"
    return "no_negation"

negation_rows = []
pattern_counts = Counter()
cue_counts = Counter()

for idx, row in enumerate(dataset):
    cues1 = extract_negation_cues(row["sentence1"])
    cues2 = extract_negation_cues(row["sentence2"])
    pattern = sentence_negation_pattern(cues1, cues2)
    if pattern != "no_negation":
        enriched = dict(row)
        enriched["orig_idx"] = idx
        enriched["negation_cues_s1"] = cues1
        enriched["negation_cues_s2"] = cues2
        enriched["negation_pattern"] = pattern
        enriched["num_neg_cues_s1"] = len(cues1)
        enriched["num_neg_cues_s2"] = len(cues2)
        enriched["num_neg_cues_total"] = len(cues1) + len(cues2)
        negation_rows.append(enriched)
        pattern_counts[pattern] += 1
        cue_counts.update(cues1)
        cue_counts.update(cues2)

print(f"Negation-focused subset size: {len(negation_rows)}")
print(f"Negation pattern counts: {pattern_counts}")
print(f"Negation cue counts: {cue_counts}")
if negation_rows:
    print("Sample negation example:")
    sample = negation_rows[0]
    print({
        "orig_idx": sample["orig_idx"],
        "label": sample["label"],
        "negation_pattern": sample["negation_pattern"],
        "negation_cues_s1": sample["negation_cues_s1"],
        "negation_cues_s2": sample["negation_cues_s2"]
    })

if len(negation_rows) == 0:
    raise ValueError("Negation-focused subset is empty. Check the regex or dataset split.")


In [ ]:
sent1_list = [r["sentence1"] for r in negation_rows]
sent2_list = [r["sentence2"] for r in negation_rows]
labels = [r["label"] for r in negation_rows]

batch_size = 32
predictions = []
confidences = []
probabilities = []

for start_idx in range(0, len(negation_rows), batch_size):
    batch_s1 = sent1_list[start_idx:start_idx + batch_size]
    batch_s2 = sent2_list[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch_s1,
        batch_s2,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)
    predictions.extend(preds.cpu().tolist())
    confidences.extend(probs.max(dim=-1).values.cpu().tolist())
    probabilities.extend(probs.cpu().tolist())

print(f"Completed inference for {len(predictions)} negation-subset examples.")


In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

per_pattern = {}
grouped_true = defaultdict(list)
grouped_pred = defaultdict(list)

for row, true_label, pred_label in zip(negation_rows, labels, predictions):
    pattern = row["negation_pattern"]
    grouped_true[pattern].append(true_label)
    grouped_pred[pattern].append(pred_label)

for pattern in ["both_sentences_negation", "sentence1_only_negation", "sentence2_only_negation"]:
    if len(grouped_true[pattern]) > 0:
        per_pattern[pattern] = {
            "count": len(grouped_true[pattern]),
            "accuracy": accuracy_score(grouped_true[pattern], grouped_pred[pattern])
        }

print("Evaluation metrics on negation-focused subset:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)
print()
print("Negation pattern counts:")
for pattern in ["both_sentences_negation", "sentence1_only_negation", "sentence2_only_negation"]:
    print(f"{pattern}: {pattern_counts.get(pattern, 0)}")
print()
print("Per-pattern accuracy:")
for pattern in ["both_sentences_negation", "sentence1_only_negation", "sentence2_only_negation"]:
    if pattern in per_pattern:
        print(f"{pattern}: count={per_pattern[pattern]['count']} | accuracy={per_pattern[pattern]['accuracy']:.4f}")
    else:
        print(f"{pattern}: count=0 | accuracy=NA")


In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}

flip_suspects = []
for i, row in enumerate(negation_rows):
    pattern = row["negation_pattern"]
    one_sided_negation = pattern in {"sentence1_only_negation", "sentence2_only_negation"}
    mismatch = predictions[i] != labels[i]
    high_conf = confidences[i] >= 0.70
    if one_sided_negation and mismatch and high_conf:
        flip_suspects.append({
            "subset_idx": i,
            "orig_idx": row["orig_idx"],
            "true_label": labels[i],
            "pred_label": predictions[i],
            "confidence": confidences[i],
            "prob_not_paraphrase": probabilities[i][0],
            "prob_paraphrase": probabilities[i][1],
            "negation_pattern": pattern,
            "negation_cues_s1": row["negation_cues_s1"],
            "negation_cues_s2": row["negation_cues_s2"],
            "sentence1": row["sentence1"],
            "sentence2": row["sentence2"]
        })

flip_suspects = sorted(flip_suspects, key=lambda x: x["confidence"], reverse=True)
print(f"Examples where negation likely flipped the decision: {len(flip_suspects)}")

num_examples_to_show = min(10, len(flip_suspects))
for i in range(num_examples_to_show):
    m = flip_suspects[i]
    print(f"Example {i + 1}")
    print(f"orig_idx: {m['orig_idx']} | negation_pattern: {m['negation_pattern']}")
    print(f"true label: {m['true_label']} ({label_map[m['true_label']]})")
    print(f"pred label: {m['pred_label']} ({label_map[m['pred_label']]})")
    print(f"confidence: {m['confidence']:.4f}")
    print(f"p(not_paraphrase)={m['prob_not_paraphrase']:.4f} | p(paraphrase)={m['prob_paraphrase']:.4f}")
    print(f"negation_cues_s1: {m['negation_cues_s1']}")
    print(f"negation_cues_s2: {m['negation_cues_s2']}")
    print(f"sentence1: {m['sentence1']}")
    print(f"sentence2: {m['sentence2']}")
    print("-" * 100)


In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print("subset=negation-focused subset")
print(f"device={device}")
print(f"num_examples_full={len(dataset)}")
print(f"num_examples_subset={len(negation_rows)}")
print(f"both_sentences_negation_count={pattern_counts.get('both_sentences_negation', 0)}")
print(f"sentence1_only_negation_count={pattern_counts.get('sentence1_only_negation', 0)}")
print(f"sentence2_only_negation_count={pattern_counts.get('sentence2_only_negation', 0)}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
for pattern in ["both_sentences_negation", "sentence1_only_negation", "sentence2_only_negation"]:
    if pattern in per_pattern:
        print(f"{pattern}_accuracy={per_pattern[pattern]['accuracy']:.4f}")
    else:
        print(f"{pattern}_accuracy=NA")
print(f"num_negation_flip_suspects={len(flip_suspects)}")
